# Run the numbers

Put your filled-in field sheet next to this notebook, point at it below, run the cell.

**You need:** a field sheet (one row per stem). **Helpful but optional:** a grid sheet, so
the answer covers a whole area instead of just the plots you measured.

Nothing to install. If something goes wrong, the output says which file, which row and which
column, so you know where to look.

## 1. Point at files

In [1]:
fieldSheet = 'sample_data/field_sheet_clean.csv'  #INPUT path to filled-in field sheet
gridSheet  = 'sample_data/grid_sheet.csv'         #INPUT `None` or grid sheet path

# What each measured plot came to, grouped by class. Fill this in as plots come back.
# Two plots per class minimum, or there is no honest error bar.
plotTotals = {                                    #INPUT t/ha from each plot
    'thick':  [52.1, 44.8, 61.3],
    'medium': [27.4, 31.9, 22.6],
    'thin':   [9.8, 12.2, 7.1]}

## 2. Run it

In [2]:
import run_pipeline

result = run_pipeline.run(
    field_sheet=fieldSheet,
    grid_sheet=gridSheet,
    class_plot_totals=plotTotals,
)

print(result.report())


--- WHAT WAS READ -------------------------------------------------------
  STOPPED: field sheet not found at 'sample_data/field_sheet_clean.csv'. Check the spelling, and that the file is where you think it is. (could not list '/Users/tahreemk/Downloads/sample_data')

--- CHECKS ON THE SHEET -------------------------------------------------
  not run

--- NO ANSWER -----------------------------------------------------------
  The sheet did not pass. Fix the errors above and run again.
  Nothing below this line was calculated.


## If it stopped instead of answering

Read the **WHAT WAS READ** and **CHECKS ON THE SHEET** parts at the top. Between them they
name the file, the row and the column.

| What it says | What to do |
|---|---|
| not found at '...' | Wrong name or wrong folder. It lists the spreadsheets it can see |
| missing required column | A heading is misspelled or missing. Headings are case sensitive and go on row 1 |
| d30_cm outside 0.3 to 120 | A decimal point moved. 450 should be 45.0 or 4.50 |
| is not a number | Units typed in the cell, like `6.3cm`. Put only the number |
| already appears on row N | Same stem written twice, or a stem number reused |
| differs across the plant's rows | Crown or height typed differently on two rows of one plant |
| UNEXPECTED PROBLEM | A real bug. Send the trace to whoever maintains the scripts |

Fix the sheet, save it, run Step 2 again.

## 3. Change the assumptions

Three of the numbers are not measured yet, so they are set by hand. Change them here and
re-run to see how much they matter.

**The simple way** is one dial for each stage:

- `residue_fraction` is how much of the standing wood you can actually use, all in one number
- `biochar_ratio` is tonnes of biochar per tonne of residue, from your kiln

Set them to whatever your own experience says.

In [ ]:
simple = run_pipeline.run(
    field_sheet=fieldSheet,
    grid_sheet=gridSheet,
    class_plot_totals=plotTotals,
    residue_fraction=0.22,        #INPUT share of standing wood you can use
    biochar_ratio=0.30,           #INPUT t biochar per t residue, your kiln
    char_carbon_fraction=0.76,    #INPUT carbon share of your char, lab tested
)
a = simple.answer
print(f"standing biomass     {a['standing_t']:>10,.0f} t")
print(f"recoverable residue  {a['residue_t']:>10,.0f} t")
print(f"biochar              {a['biochar_t']:>10,.0f} t")
print(f"CO2e                 {a['co2e_t']:>10,.0f} t")

### *Once you have weighed some kiln runs*

Stop guessing the ratio and fit it from the real runs. Yield is then an average of what your
kiln actually did, and the spread between runs travels with the answer.

`h_c_organic` is the one that decides whether the carbon counts as durable. Under 0.70 and it
qualifies; at or above, it does not, and the output will say so.

In [ ]:
fromKiln = run_pipeline.run(
    field_sheet=fieldSheet,
    grid_sheet=gridSheet,
    class_plot_totals=plotTotals,
    kiln_batches=[                #INPUT one row per weighed run
        {'dry_feedstock_t': 4.10, 'char_t': 1.19, 'carbon_fraction': 0.78, 'h_c_organic': 0.41},
        {'dry_feedstock_t': 3.85, 'char_t': 1.08, 'carbon_fraction': 0.76, 'h_c_organic': 0.44},
        {'dry_feedstock_t': 4.40, 'char_t': 1.34, 'carbon_fraction': 0.79, 'h_c_organic': 0.39},
    ],
)
for name, value, how in fromKiln.steps[-1:]:
    print(name); print(value); print(how)
print()
print(f"CO2e  {fromKiln.answer['co2e_t']:,.0f} t")

---

### Going deeper

This notebook gives you the whole-area number. To poke at single values — what one stem
weighs, what happens at the 18 cm ceiling, how one plant adds up — open
**`prosopis_residue_calculator.ipynb`**. Same maths, one number at a time.

### The parts

| File | What it does |
|---|---|
| `run_pipeline.py` | Joins everything up. This notebook just calls it |
| `field_data.py` | Reads and checks the sheet |
| `prosopis_residue.py` | Diameters to weight, weight to carbon |
| `stratify.py` | A few plots scaled up to a whole area |

You should not need to open any of them. They are there so the numbers can be checked, and so
the same code can run automatically later without being rewritten.
